# Fantasy Football Weekly Projections (Half-PPR)

Run via **papermill** from the project root:
```bash
papermill fantasy/predict_fantasy.ipynb /tmp/out.ipynb -p TARGET_SEASON 2025 -p TARGET_WEEK 14
```
Or run all cells interactively set parameters in the Parameters cell.

## Parameters

In [17]:
TARGET_SEASON = 2025
TARGET_WEEK   = 10  # None = auto-detect next unplayed week
POS_FILTER    = None  # None | "QB" | "RB" | "WR" | "TE"

## Setup — Imports & Config

In [18]:
import warnings
import joblib
import numpy as np
import pandas as pd
import nflreadpy as nfl
from pathlib import Path

warnings.filterwarnings("ignore")

# Works whether kernel starts from project root or from fantasy/ directly
_cwd         = Path.cwd()
_DIR         = _cwd if _cwd.name == "fantasy" else _cwd / "fantasy"
FEATURES_CSV = _DIR / "features_dataset.csv"
MODEL_DIR    = _DIR / "models"
POSITIONS    = ["QB", "RB", "WR", "TE"]

INJURY_MAP   = {"Out": 0.0, "Doubtful": 0.1, "Questionable": 0.5, "Probable": 0.75}
PRACTICE_MAP = {"Did Not Participate In Practice": 0.0, "Limited Participation in Practice": 0.5, "Full Participation in Practice": 1.0}

TURF_SURFACES = {"astroturf", "fieldturf", "turf", "matrixturf", "sportturf", "astroplay", "a_turf"}

## Step 1 — Load Models

In [19]:
# â”€â”€ Load models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
models = {}
for pos in POSITIONS:
    saved = joblib.load(MODEL_DIR / f"{pos.lower()}_model.pkl")
    models[pos] = saved
print("Models loaded:", {p: len(m["feature_cols"]) for p, m in models.items()})

# â”€â”€ Determine target season / week â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

Models loaded: {'QB': 58, 'RB': 80, 'WR': 80, 'TE': 80}


## Step 2 — Detect Target Week

In [20]:
def detect_week(season):
    raw   = nfl.load_schedules([season])
    sched = raw.to_pandas() if hasattr(raw, "to_pandas") else pd.DataFrame(raw)
    reg   = sched[(sched["season"] == season) & (sched["game_type"] == "REG")]
    future = reg[reg["result"].isna()]
    return int(future["week"].min()) if not future.empty else None


if TARGET_WEEK is None:
    TARGET_WEEK = detect_week(TARGET_SEASON)
if TARGET_WEEK is None:
    raise ValueError(f"Season {TARGET_SEASON} complete â€” no upcoming games.")

print(f"\nProjecting Season {TARGET_SEASON}  Week {TARGET_WEEK}"
      + (f"  [{POS_FILTER}]" if POS_FILTER else "  [all positions]"))


Projecting Season 2025  Week 10  [all positions]


## Step 3 — Upcoming Schedule

In [21]:
# â”€â”€ Load upcoming schedule â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
raw_sched = nfl.load_schedules([TARGET_SEASON])
schedule  = raw_sched.to_pandas() if hasattr(raw_sched, "to_pandas") else pd.DataFrame(raw_sched)
schedule["season"] = schedule["season"].astype(int)
schedule["week"]   = schedule["week"].astype(int)

upcoming = schedule[
    (schedule["season"] == TARGET_SEASON) &
    (schedule["week"]   == TARGET_WEEK) &
    (schedule["game_type"] == "REG")
].copy()

if upcoming.empty:
    raise ValueError(f"No REG games found for week {TARGET_WEEK}.")

print(f"{len(upcoming)} games found.")

# Build one row per team per game
home = upcoming[["game_id", "home_team", "away_team", "spread_line", "total_line",
                  "roof", "surface", "temp", "wind", "home_rest", "away_rest", "gameday"]].copy()
home.rename(columns={"home_team": "team", "away_team": "opponent_team", "home_rest": "rest"}, inplace=True)
home["is_home"] = 1

away = upcoming[["game_id", "away_team", "home_team", "spread_line", "total_line",
                  "roof", "surface", "temp", "wind", "home_rest", "away_rest", "gameday"]].copy()
away.rename(columns={"away_team": "team", "home_team": "opponent_team", "away_rest": "rest"}, inplace=True)
away["is_home"] = 0

team_ctx = pd.concat([home, away], ignore_index=True)

team_ctx["temp"]  = team_ctx["temp"].fillna(72)
team_ctx["wind"]  = team_ctx["wind"].fillna(0)
team_ctx["is_dome"]  = team_ctx["roof"].isin(["dome", "closed"]).astype(int)
team_ctx["is_turf"]  = team_ctx["surface"].isin(TURF_SURFACES).astype(int)
team_ctx["effective_wind"] = np.where(team_ctx["is_dome"], 0,  team_ctx["wind"])
team_ctx["effective_temp"] = np.where(team_ctx["is_dome"], 72, team_ctx["temp"])
team_ctx["days_rest"] = team_ctx["rest"].fillna(7)

# implied_team_total: home = (total - spread) / 2, away = (total + spread) / 2
# spread_line is home-perspective (negative = home favored), same as features_dataset
team_ctx["implied_team_total"] = np.where(
    team_ctx["is_home"] == 1,
    (team_ctx["total_line"] - team_ctx["spread_line"]) / 2,
    (team_ctx["total_line"] + team_ctx["spread_line"]) / 2,
)

ctx_cols = ["team", "opponent_team", "is_home", "spread_line", "total_line",
            "implied_team_total", "days_rest", "is_dome", "is_turf",
            "effective_wind", "effective_temp", "gameday", "game_id"]

14 games found.


## Step 4 — Player History

In [22]:
# â”€â”€ Load player history â€” each player's latest rolling form â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
hist = pd.read_csv(FEATURES_CSV)
hist["season"] = hist["season"].astype(int)
hist["week"]   = hist["week"].astype(int)

# Most recent completed row per player = current rolling form
latest = (
    hist.sort_values(["player_id", "season", "week"])
    .groupby("player_id").last().reset_index()
)

# Drop players who haven't appeared in the last two seasons (retired/cut)
latest = latest[latest["season"] >= TARGET_SEASON - 1]

# Latest opponent defensive metrics per team (for upcoming opponent lookup)
DEF_COLS = ["def_epa_allowed_roll4", "def_yards_allowed_roll4",
            "def_pass_rate_faced_roll4", "def_red_zone_allowed_roll4"]
opp_def = (
    hist[["opponent_team", "season", "week"] + DEF_COLS]
    .sort_values(["opponent_team", "season", "week"])
    .groupby("opponent_team").last().reset_index()
    [["opponent_team"] + DEF_COLS]
)

# â”€â”€ Join players with upcoming game context â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
active_teams = team_ctx["team"].tolist()
players = latest[latest["team"].isin(active_teams)].copy()
if POS_FILTER:
    players = players[players["position"] == POS_FILTER]

drop_cols = [c for c in ctx_cols + DEF_COLS if c != "team"]
players.drop(columns=drop_cols, errors="ignore", inplace=True)

players = players.merge(team_ctx[ctx_cols], on="team", how="inner")
players = players.merge(opp_def, on="opponent_team", how="left")
print(f"Players matched to upcoming games: {len(players)}")

Players matched to upcoming games: 963


## Step 5 — Injury & Depth Chart Refresh

In [23]:
# â”€â”€ Refresh injuries for this week â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
try:
    raw_inj = nfl.load_injuries(seasons=[TARGET_SEASON])
    inj     = raw_inj.to_pandas() if hasattr(raw_inj, "to_pandas") else pd.DataFrame(raw_inj)
    inj_wk  = inj[inj["week"] == TARGET_WEEK].copy()
    inj_wk["injury_status_score"]   = inj_wk["report_status"].map(INJURY_MAP).fillna(1.0)
    inj_wk["practice_status_score"] = inj_wk["practice_status"].map(PRACTICE_MAP).fillna(1.0)
    inj_wk = inj_wk[["gsis_id", "injury_status_score", "practice_status_score"]].rename(
        columns={"gsis_id": "player_id"}
    )
    players.drop(columns=["injury_status_score", "practice_status_score"], errors="ignore", inplace=True)
    players = players.merge(inj_wk, on="player_id", how="left")
    players["injury_status_score"]   = players["injury_status_score"].fillna(1.0)
    players["practice_status_score"] = players["practice_status_score"].fillna(1.0)
    print("Injuries updated.")
except Exception as e:
    print(f"Injuries unavailable ({e}) â€” using last known values")

# ── Refresh depth charts ────────────────────────────────────────────
try:
    raw_dc = nfl.load_depth_charts(seasons=[TARGET_SEASON])
    dc     = raw_dc.to_pandas() if hasattr(raw_dc, "to_pandas") else pd.DataFrame(raw_dc)
    # nflreadpy uses a dt timestamp, not season/week — take the most recent snapshot
    dc["dt"] = pd.to_datetime(dc["dt"], utc=True)
    latest_dt = dc["dt"].max()
    dc_latest = dc[dc["dt"] == latest_dt].copy()
    # pos_abb is the position (QB/RB/WR/TE), pos_rank is depth (1=starter)
    dc_skill = dc_latest[dc_latest["pos_abb"].isin(["QB", "RB", "WR", "TE"])]
    dc_clean = (
        dc_skill
        .sort_values("pos_rank")
        .drop_duplicates(subset=["gsis_id"], keep="first")
        [["gsis_id", "pos_rank"]]
        .rename(columns={"gsis_id": "player_id", "pos_rank": "depth_chart_position"})
    )
    players.drop(columns=["depth_chart_position"], errors="ignore", inplace=True)
    players = players.merge(dc_clean, on="player_id", how="left")
    players["depth_chart_position"] = players["depth_chart_position"].fillna(2)
    print(f"Depth charts updated from snapshot: {latest_dt.date()}")
except Exception as e:
    print(f"Depth charts unavailable ({e}) — using last known values")
# ── Recompute team rankings from current rolling values ───────────────────────
# Use most recent team-level row from hist so all players on the same team
# share one authoritative value, then rank within this week's matchups.
team_metrics = (
    hist[['team', 'season', 'week', 'off_epa_roll4', 'opp_win_pct_roll4']]
    .drop_duplicates(subset=['team', 'season', 'week'])
    .sort_values(['team', 'season', 'week'])
    .groupby('team').last()
    .reset_index()[['team', 'off_epa_roll4', 'opp_win_pct_roll4']]
    .rename(columns={'off_epa_roll4': '_epa', 'opp_win_pct_roll4': '_sos'})
)
active = team_metrics[team_metrics['team'].isin(players['team'].unique())].copy()
active['off_epa_rank'] = active['_epa'].rank(ascending=False, method='min').astype(int)
active['sos_rank']     = active['_sos'].rank(ascending=False, method='min').astype(int)

players.drop(columns=['off_epa_roll4', 'opp_win_pct_roll4', 'off_epa_rank', 'sos_rank'], errors='ignore', inplace=True)
players = players.merge(active[['team', '_epa', '_sos', 'off_epa_rank', 'sos_rank']], on='team', how='left')
players.rename(columns={'_epa': 'off_epa_roll4', '_sos': 'opp_win_pct_roll4'}, inplace=True)
print(f"Rankings recomputed across {len(active)} teams playing this week.")

# Drop players officially ruled Out
before = len(players)
players = players[players["injury_status_score"] > 0.0]
print(f"Dropped {before - len(players)} players ruled Out.")

Injuries unavailable ('practice_primary_status') â€” using last known values
Depth charts updated from snapshot: 2026-03-14


## Step 6 — Generate Projections

In [ ]:
# â”€â”€ Score with position models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
all_proj = []

for pos in POSITIONS:
    if POS_FILTER and pos != POS_FILTER:
        continue
    pos_players = players[players["position"] == pos].copy()
    if pos_players.empty:
        continue

    feat_cols = models[pos]["feature_cols"]
    missing   = [c for c in feat_cols if c not in pos_players.columns]
    if missing:
        print(f"  {pos}: {len(missing)} features missing â€” filling 0: {missing[:5]}")
        for c in missing:
            pos_players[c] = 0

    X = pos_players[feat_cols].fillna(pos_players[feat_cols].median())
    pos_players = pos_players.copy()
    pos_players["projected_pts"] = models[pos]["model"].predict(X).round(2)
    all_proj.append(pos_players)

if not all_proj:
    raise ValueError("No projections generated, check that features_dataset.csv is up to date.")

proj = pd.concat(all_proj, ignore_index=True)

# â”€â”€ Print + save â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

print(f"\n{'='*65}")
print(f"  Season {TARGET_SEASON}  Week {TARGET_WEEK}  Half-PPR Projections")
print(f"{'='*65}")

for pos in POSITIONS:
    if POS_FILTER and pos != POS_FILTER:
        continue
    subset = (
        proj[proj["position"] == pos]
        .sort_values("projected_pts", ascending=False)
        .head(20)
        [["player_display_name", "team", "opponent_team", "projected_pts",
          "implied_team_total", "depth_chart_position", "injury_status_score"]]
        .reset_index(drop=True)
    )
    if subset.empty:
        continue
    subset.index += 1
    print(f"\n--- {pos} (Top 20) ---")
    print(subset.to_string())

out_path = _DIR / f"projections_{TARGET_SEASON}_week{TARGET_WEEK:02d}.csv"
proj[["player_id", "player_display_name", "position", "team", "opponent_team",
      "gameday", "projected_pts", "implied_team_total",
      "depth_chart_position", "injury_status_score", "is_home", "off_epa_roll4", "opp_season_win_pct", "opp_win_pct_roll4", "off_epa_rank", "sos_rank"]].sort_values(
    ["position", "projected_pts"], ascending=[True, False]
).to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")


  Season 2025  Week 10  â€”  Half-PPR Projections

--- QB (Top 20) ---
   player_display_name team opponent_team  projected_pts  implied_team_total  depth_chart_position  injury_status_score
1          Brock Purdy   SF            LA      23.059999               27.50                   1.0                  1.0
2           Josh Allen  BUF           MIA      22.850000               21.00                   1.0                  1.0
3       Jayden Daniels  WAS           DET      21.580000               29.00                   1.0                  1.0
4          Jordan Love   GB           PHI      20.870001               22.00                   1.0                  1.0
5          Jalen Hurts  PHI            GB      20.780001               23.50                   1.0                  1.0
6               Bo Nix  DEN            LV      20.570000               16.50                   1.0                  1.0
7     Matthew Stafford   LA            SF      20.080000               22.00            

## Step 7 — Save Output